# Lab Day 1 — Notebook

> **Quy trình làm bài**
> 1. Viết các module trực tiếp trong `code/` bằng VS Code.
> 2. Đồng bộ lên GitHub bằng commit/push; trên Colab cập nhật repo bằng git pull trước khi chạy.
> 3. Ô thiết lập bên dưới dùng `REPO/code/` để chạy, lưu ảnh/kết quả vào `submission_<MSSV>/`.
> 4. Cuối bài, copy code hoàn thiện vào `submission_<MSSV>/code/`, lưu notebook có output vào đó.
> 5. Sau khi hoàn thiện các TODO: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
from pathlib import Path

MSSV = "2A202602457"
REPO_URL = "https://github.com/sown101/K4-Track4-Day1-Neuralnetwork.git"

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO = next((p for p in candidates if (p / "data").is_dir() and (p / "scripts").is_dir() and ((p / "code/lab.ipynb").is_file() or (p / f"submission_{MSSV}/code/lab.ipynb").is_file())), None)
    if REPO is None:
        raise FileNotFoundError("Hãy mở notebook từ thư mục dự án.")
else:
    IN_COLAB = True
    drive.mount("/content/drive")
    REPO = Path("/content/drive/MyDrive/K4-Track4-Day1-Neuralnetwork")
    if not REPO.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
    # Cập nhật code mới nhất đã push từ máy cá nhân.
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)

REPO = REPO.resolve()
SUBMISSION = REPO / f"submission_{MSSV}"
# Bản nộp chỉ có code riêng: ưu tiên dùng code đã đóng gói.
CODE_DIR = SUBMISSION / "code" if not (REPO / "code").is_dir() else REPO / "code"
for name in ("figures", "results"):
    (SUBMISSION / name).mkdir(parents=True, exist_ok=True)
os.chdir(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
REPO_ROOT = str(REPO)
OUT_DIR = str(SUBMISSION)

import numpy as np, torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if IN_COLAB and device != "cuda":
    raise RuntimeError("Hãy bật GPU trong Runtime trước khi tiếp tục.")
print("PyTorch:", torch.__version__)
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Thư mục code:", CODE_DIR)
print("Thư mục kết quả:", OUT_DIR)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx
from runner import run_or_load

# LAB_SMOKE=1: chạy thử nhanh pipeline (ít dữ liệu, ít epoch) — KHÔNG dùng cho kết quả nộp
SMOKE = os.environ.get("LAB_SMOKE") == "1"
FORCE_RERUN = False      # True: bỏ qua kết quả JSON đã lưu và huấn luyện lại tất cả
CKPT_DIR = str(REPO / "checkpoints")   # trọng số baseline/final để dự đoán eval; không nộp
%matplotlib inline

## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation (20% của train, phân tầng, seed 42), chuẩn hoá 10 cột số bằng thống kê **chỉ của phần train còn lại**, đưa toàn bộ lên device một lần.

In [ ]:
processed_dir = REPO / "data" / "processed"
if not all((processed_dir / f"{name}.npz").is_file() for name in ("train", "eval")):
    completed = subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True, capture_output=True, text=True)
    print(completed.stdout)
for name in ("train", "eval"):
    with np.load(processed_dir / f"{name}.npz") as dataset:
        print(name, "X:", dataset["X"].shape, dataset["X"].dtype, "y:", dataset["y"].shape, dataset["y"].dtype, "nhãn:", dataset["y"].min(), "..", dataset["y"].max())

data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")
assert len(data["X_tr"]) == 371_847 and len(data["X_val"]) == 92_962 and len(data["X_eval"]) == 116_203
num = data["X_tr"][:, :10]
print("mean 10 cột số (X_tr):", num.mean(0).cpu().numpy().round(4))
print("std  10 cột số (X_tr):", num.std(0).cpu().numpy().round(4))

if SMOKE:  # chỉ để kiểm tra code chạy được trên CPU
    g = torch.Generator().manual_seed(0)
    for k, n in (("tr", 20_000), ("val", 5_000)):
        idx = torch.randperm(len(data[f"X_{k}"]), generator=g)[:n].to(device)
        data[f"X_{k}"], data[f"y_{k}"] = data[f"X_{k}"][idx], data[f"y_{k}"][idx]
    print("SMOKE: dùng tập con", len(data["X_tr"]), len(data["X_val"]))

**Nhận xét Part 0:** kích thước đúng quy định (371 847 / 92 962 / 116 203); tỉ lệ lớp ở train, val, eval gần như trùng nhau nhờ phân tầng;
10 cột số sau chuẩn hoá có mean ≈ 0, std ≈ 1 trên X_tr. Mốc thấp nhất "luôn đoán lớp 1" trên val ≈ 0,4876.
Mean/std chỉ tính trên phần train còn lại; X_eval chỉ được *áp dụng* phép chuẩn hoá đó, không tham gia tính thống kê, chọn cấu hình hay dừng sớm.

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
import math
import matplotlib.pyplot as plt
import torch.nn.functional as F

set_seed(1)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_params = count_params(model)
assert n_params == EXPECTED_PARAMS[(256, 128)], n_params
print(model)
print("số tham số:", n_params)
for k, (h, n) in enumerate(EXPECTED_PARAMS.items()):
    assert count_params(MLP(hidden=h)) == n
print("M-wide / M-deep:", count_params(MLP(hidden=(512, 256))), "/", count_params(MLP(hidden=(256, 128, 64))))

# 1b. shape qua từng lớp
x = torch.randn(8, 54, device=device)
h = x
for layer in model.net:
    h = layer(h)
    print(f"{layer.__class__.__name__:8s} -> {tuple(h.shape)}")
assert model(x).shape == (8, 7)

# 1c-1. loss bước 0 trên toàn bộ val (eval mode)
step0 = evaluate(model, data["X_val"], data["y_val"])
print(f"\nloss bước 0 trên val = {step0['loss']:.4f}   (ln 7 = {math.log(7):.4f})")
print("độ lệch chuẩn kích hoạt sau mỗi ReLU / logit (He):", [round(s, 3) for s in activation_stats(model, data["X_val"][:4096])])

In [ ]:
# 1c-2. quá khớp 20 mẫu (dropout tắt): loss phải về gần 0
set_seed(0)
tiny = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
idx = torch.randperm(len(data["X_tr"]), generator=torch.Generator().manual_seed(0))[:20].to(device)
xb, yb = data["X_tr"][idx], data["y_tr"][idx]
opt = build_optimizer("adam", tiny.parameters(), lr=1e-3)
curve = []
for step in range(500):
    tiny.train()
    loss = F.cross_entropy(tiny(xb), yb)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    curve.append(loss.item())
acc20 = (predict(tiny, xb) == yb).float().mean().item()
print(f"loss ban đầu {curve[0]:.4f} -> sau 500 bước {curve[-1]:.6f};  accuracy trên 20 mẫu = {acc20:.2f}")
print("các nhãn trong 20 mẫu:", torch.bincount(yb, minlength=7).tolist())
plt.figure(figsize=(5, 3.2))
plt.semilogy(curve); plt.xlabel("bước"); plt.ylabel("CE loss (log)"); plt.title("Quá khớp 20 mẫu (Adam lr=1e-3)"); plt.grid(alpha=.3)
plt.show()
assert curve[-1] < 1e-2 and acc20 == 1.0

# 1d. gradient có chảy tới mọi tham số?
model.train()
loss = F.cross_entropy(model(data["X_tr"][:512]), data["y_tr"][:512])
model.zero_grad(); loss.backward()
names = ["W1", "b1", "W2", "b2", "W3", "b3"]
for nm, (pname, p) in zip(names, model.named_parameters()):
    gn = p.grad.norm().item() if p.grad is not None else None
    print(f"{nm} ({pname:12s}) shape {tuple(p.shape)!s:12s} ‖grad‖ = {gn:.5f}")
    assert p.grad is not None and gn > 0

**Nhận xét Part 1:**
- Số tham số đúng 47 879 (assert), logits `(B, 7)`; softmax không nằm trong model mà trong `F.cross_entropy`.
- Loss bước 0 đo được (xem output) rất gần ln 7 ≈ 1,946: với khởi tạo He và bias = 0, logit ở bước 0 nhỏ và gần như đối xứng giữa 7 lớp → softmax ≈ đều 1/7 → CE ≈ ln 7. Nếu lệch nhiều thì là dấu hiệu logit quá lớn (khởi tạo/chuẩn hoá sai).
- Quá khớp 20 mẫu: loss giảm về gần 0 và accuracy 100% trên 20 mẫu → vòng lặp (nhãn, loss, `zero_grad`, optimizer nhận tham số) không có lỗi cơ bản.
- Cả 6 tham số W1, b1, W2, b2, W3, b3 đều có gradient khác 0 → gradient chảy tới mọi lớp.

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` (trong `train.py`) nhận một dict cấu hình và trả về lịch sử + tóm tắt. `run_or_load` (trong `runner.py`) gọi nó,
lưu `results/<exp_id>.json` và ảnh `figures/<exp_id>.png`. Baseline: SGD+momentum 0,9, CE, He, batch 512, 20 epoch, dropout 0, không clip, FP32.

Mỗi epoch ghi: train loss (eval mode, 50 000 mẫu train cố định), val loss/acc/macro-F1, grad_norm trung bình và lớn nhất (trước clip), tỉ lệ bước bị clip,
thời gian epoch; cùng loss bước 0, best epoch theo val loss, bộ nhớ GPU cực đại, cờ diverged.

**Chọn lr cho baseline bằng VAL.** Dự đoán: lr = 0,01 sẽ chậm (20 epoch chưa đủ), lr quá lớn (0,3, tương đương bước hiệu dụng lr/(1−μ) = 3) có thể dao động;
giá trị tốt nhất có lẽ quanh 0,03–0,1. Các lần chạy này cũng là thí nghiệm "độ nhạy lr" của SGD+momentum (nhóm `optimizer`).

In [ ]:
import pandas as pd

def R(cfg, keep_state=False, show=True):
    """Chạy (hoặc nạp lại) một cấu hình. SMOKE: 2 epoch để kiểm tra code."""
    if SMOKE:
        cfg = {**cfg, "epochs": min(cfg.get("epochs", 20), 2)}
    return run_or_load(cfg, data, OUT_DIR, ckpt_dir=CKPT_DIR, keep_state=keep_state, force=FORCE_RERUN, show=show)

def table(results, base_mean=None, noise=None):
    rows = []
    for r in results:
        s, c = r["summary"], r["cfg"]
        row = dict(exp_id=c["exp_id"], lr=c["lr"], step0=s["step0_loss"], best_ep=s["best_epoch"],
                   best_val_loss=s["best_val_loss"], train_loss=s["final_train_loss"], val_loss=s["final_val_loss"],
                   val_acc=s["val_acc"], val_F1=s["val_macro_f1"], s_per_ep=s["time_per_epoch_s"],
                   mem_MB=s["peak_mem_MB"], diverged=s["diverged"])
        if base_mean is not None and s["val_macro_f1"] is not None:
            row["ΔF1_vs_base"] = s["val_macro_f1"] - base_mean
            if noise is not None:
                row["vượt 2σ?"] = "Có" if abs(row["ΔF1_vs_base"]) > noise else "Không"
        rows.append(row)
    df = pd.DataFrame(rows).set_index("exp_id")
    with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 200):
        display(df)
    return df

BASE = {**DEFAULT_CFG}
LR_GRID = [0.01, 0.03, 0.1, 0.3]
sweep = [R({**BASE, "exp_id": f"opt-sgdm-lr{lr:g}", "group": "optimizer", "lr": lr,
            "description": f"SGD+momentum 0.9, lr={lr:g} (dò lr cho baseline)"}, show=False) for lr in LR_GRID]
df_sweep = table(sweep)
ok = [r for r in sweep if not r["summary"]["diverged"]]
BASE_LR = max(ok, key=lambda r: r["summary"]["val_macro_f1"])["cfg"]["lr"]
print("lr chọn cho baseline (theo val macro-F1):", BASE_LR)
plot_compare(sweep, ["val_loss", "val_macro_f1", "grad_norm"], f"{OUT_DIR}/figures/compare_baseline_lr.png",
             "SGD+momentum: dò lr cho baseline (val)")
from IPython.display import Image, display
display(Image(f"{OUT_DIR}/figures/compare_baseline_lr.png", width=950))

In [ ]:
# Baseline với 3 seed để đo độ nhiễu. base-s1 trùng cấu hình với opt-sgdm-lr<BASE_LR> -> kiểm tra tái lập.
BASE = {**DEFAULT_CFG, "lr": BASE_LR}
base_runs = []
for s in (1, 2, 3):
    base_runs.append(R({**BASE, "seed": s, "exp_id": f"base-s{s}", "group": "baseline",
                        "description": f"Baseline M-base, seed {s}"}, keep_state=(s == 1), show=(s == 1)))
df_base = table(base_runs)
f1s = np.array([r["summary"]["val_macro_f1"] for r in base_runs])
accs = np.array([r["summary"]["val_acc"] for r in base_runs])
BASE_F1, BASE_SD = f1s.mean(), f1s.std(ddof=1)
NOISE = 2 * BASE_SD
print(f"val macro-F1 baseline: {BASE_F1:.4f} ± {BASE_SD:.4f} (σ mẫu, 3 seed)  -> ngưỡng nhiễu 2σ = {NOISE:.4f}")
print(f"val accuracy baseline: {accs.mean():.4f} ± {accs.std(ddof=1):.4f}")
same = [r for r in sweep if r["cfg"]["lr"] == BASE_LR][0]["summary"]["val_macro_f1"]
print(f"tái lập: opt-sgdm-lr{BASE_LR:g} F1 = {same:.4f}  vs  base-s1 F1 = {base_runs[0]['summary']['val_macro_f1']:.4f}")
plot_compare(base_runs, ["train_loss", "val_loss", "val_macro_f1"], f"{OUT_DIR}/figures/compare_baseline_seeds.png",
             "Baseline: 3 seed (độ nhiễu)")
display(Image(f"{OUT_DIR}/figures/compare_baseline_seeds.png", width=950))

**Nhận xét baseline:** _(xem mục Đối chiếu sau khi chạy ở cuối Part 2)_

## Part 3 — Thí nghiệm
Mỗi nhóm: **dự đoán trước** (viết trước khi chạy) → đổi **một** yếu tố so với baseline (cùng split, cùng 20 epoch, seed 1) → chạy → **đối chiếu**.
So sánh luôn bằng val macro-F1 của epoch có val loss thấp nhất, với ngưỡng nhiễu 2σ đo ở Part 2. Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Chủ đề 1 — Hàm mất mát: CE vs MSE (`loss-*`)
**Dự đoán trước:** MSE trên one-hot (trung bình trên mọi B·7 phần tử) cho gradient theo logit `2(z − y)/7` — bị chặn và nhỏ khi dự đoán sai nặng, trong khi CE cho `softmax(z) − y` không bão hoà.
Với cùng lr, MSE sẽ học chậm hơn rõ rệt và macro-F1 thấp hơn baseline (nhất là các lớp hiếm, vì tín hiệu gradient của chúng vốn đã nhỏ). Tăng lr ×3 cho MSE sẽ thu hẹp một phần khoảng cách.
Không so giá trị loss CE với MSE (khác thang đo), chỉ so accuracy / macro-F1 và tốc độ hội tụ.

In [ ]:
loss_runs = [
    R({**BASE, "exp_id": "loss-mse", "group": "loss", "loss": "mse", "description": "MSE trên one-hot (mean trên B*7), cùng lr baseline"}),
    R({**BASE, "exp_id": "loss-mse-lrx3", "group": "loss", "loss": "mse", "lr": 3 * BASE_LR,
       "description": "MSE, lr x3 (đổi 2 yếu tố: loss + lr, để bù gradient nhỏ của MSE)"}, show=False),
]
table([base_runs[0]] + loss_runs, BASE_F1, NOISE)
plot_compare([base_runs[0]] + loss_runs, ["val_acc", "val_macro_f1"], f"{OUT_DIR}/figures/compare_loss.png", "CE vs MSE (so bằng metric, không so loss)")
display(Image(f"{OUT_DIR}/figures/compare_loss.png", width=800))

**Đối chiếu (loss):** _điền sau khi chạy_

### Chủ đề 2 — Bộ tối ưu hoá (`opt-*`)
**Dự đoán trước:** SGD không momentum cần lr lớn hơn ~10 lần SGD+momentum mới đi được quãng đường tương đương (bước hiệu dụng của momentum ≈ lr/(1−μ) = 10·lr), nên ở cùng lr nó chậm hơn.
Adam/AdamW chuẩn hoá bước theo √v̂ của từng tham số nên ít nhạy với tỉ lệ gradient và hội tụ nhanh hơn ở vài epoch đầu; ở lr tốt nhất của mỗi bộ, Adam có thể nhỉnh hơn SGD+momentum
nhưng khác biệt có thể không vượt 2σ. AdamW với `weight_decay=0` phải cho kết quả **trùng** Adam (cùng seed).
Mỗi bộ thử 3 lr cách nhau ~3 lần, so ở lr tốt nhất (theo val macro-F1) của từng bộ.

In [ ]:
opt_grid = {"sgd": [0.03, 0.1, 0.3], "adam": [3e-4, 1e-3, 3e-3], "adamw": [3e-4, 1e-3, 3e-3]}
short = {"sgd": "sgd", "adam": "adam", "adamw": "adamw"}
opt_runs = {"sgd_momentum": sweep}
for name, lrs in opt_grid.items():
    opt_runs[name] = [R({**BASE, "exp_id": f"opt-{short[name]}-lr{lr:g}", "group": "optimizer", "optimizer": name, "lr": lr,
                         "weight_decay": 0.01 if name == "adamw" else 0.0,
                         "description": f"{name}, lr={lr:g}" + (", wd=0.01 (tách riêng)" if name == "adamw" else "")}, show=False)
                      for lr in lrs]
adamw_wd0 = R({**BASE, "exp_id": "opt-adamw-wd0-lr0.001", "group": "optimizer", "optimizer": "adamw", "lr": 1e-3, "weight_decay": 0.0,
               "description": "AdamW wd=0, lr=1e-3: kiểm chứng trùng Adam"}, show=False)
all_opt = [r for v in opt_runs.values() for r in v] + [adamw_wd0]
df_opt = table(all_opt, BASE_F1, NOISE)

best_opt = {k: max([r for r in v if not r["summary"]["diverged"]], key=lambda r: r["summary"]["val_macro_f1"]) for k, v in opt_runs.items()}
print("\nMỗi bộ tối ưu ở lr tốt nhất của nó (theo val macro-F1):")
table(list(best_opt.values()), BASE_F1, NOISE)
a = [r for r in opt_runs["adam"] if r["cfg"]["lr"] == 1e-3][0]["summary"]
print(f"Adam lr=1e-3 F1 = {a['val_macro_f1']:.6f}  |  AdamW wd=0 lr=1e-3 F1 = {adamw_wd0['summary']['val_macro_f1']:.6f}")

plot_compare(list(best_opt.values()), ["val_loss", "val_macro_f1", "grad_norm"], f"{OUT_DIR}/figures/compare_optimizer.png",
             "Bộ tối ưu, mỗi bộ ở lr tốt nhất của nó")
plot_compare(all_opt, ["val_loss", "val_macro_f1"], f"{OUT_DIR}/figures/compare_optimizer_lr.png", "Bộ tối ưu × lr (độ nhạy lr)")
display(Image(f"{OUT_DIR}/figures/compare_optimizer.png", width=950))
display(Image(f"{OUT_DIR}/figures/compare_optimizer_lr.png", width=950))

**Đối chiếu (optimizer):** _điền sau khi chạy_

### Chủ đề 3 — Hyper-parameter (`hp-*`)
**Dự đoán trước:**
- *Batch*: cùng 20 epoch, batch 128 có 4× số bước cập nhật so với 512 → học được nhiều hơn (F1 cao hơn) nhưng mỗi epoch chậm hơn nhiều; batch 2048 chỉ có 1/4 số bước → kém hơn ở cùng lr.
  Tăng lr ×4 cho batch 2048 (quy tắc tăng lr theo lô, không khởi động) sẽ bù lại phần lớn khoảng cách, nhưng có rủi ro dao động ở đầu.
- *Độ rộng/độ sâu*: M-wide (161k tham số) và M-deep có năng lực lớn hơn; vì baseline chưa quá khớp (dự đoán) nên chúng sẽ nhỉnh hơn baseline.
- *Weight decay 1e-4*: mô hình chưa quá khớp nên gần như không đổi (trong nhiễu).
- *40 epoch*: baseline 20 epoch chưa hội tụ hẳn → 40 epoch sẽ tăng F1 rõ rệt.

In [ ]:
hp_runs = [
    R({**BASE, "exp_id": "hp-batch128", "group": "hparam", "batch": 128, "description": "batch 128 (4x số bước), cùng lr"}, show=False),
    R({**BASE, "exp_id": "hp-batch2048", "group": "hparam", "batch": 2048, "description": "batch 2048 (1/4 số bước), cùng lr"}, show=False),
    R({**BASE, "exp_id": "hp-batch2048-lrx4", "group": "hparam", "batch": 2048, "lr": 4 * BASE_LR,
       "description": "batch 2048, lr x4 (quy tắc tăng lr theo lô, không warmup) — đổi 2 yếu tố có chủ đích"}, show=False),
    R({**BASE, "exp_id": "hp-wide", "group": "hparam", "hidden": (512, 256), "description": "M-wide 54-512-256-7"}, show=False),
    R({**BASE, "exp_id": "hp-deep", "group": "hparam", "hidden": (256, 128, 64), "description": "M-deep 54-256-128-64-7"}, show=False),
    R({**BASE, "exp_id": "hp-wd1e-4", "group": "hparam", "weight_decay": 1e-4, "description": "weight_decay 1e-4 (L2 trong SGD)"}, show=False),
    R({**BASE, "exp_id": "hp-ep40", "group": "hparam", "epochs": 40, "description": "40 epoch thay vì 20"}, show=False),
]
df_hp = table([base_runs[0]] + hp_runs, BASE_F1, NOISE)
print("số bước cập nhật / epoch:", {r["cfg"]["exp_id"]: r["summary"].get("steps_per_epoch") for r in [base_runs[0]] + hp_runs[:3]})
plot_compare([base_runs[0]] + hp_runs[:3], ["val_loss", "val_macro_f1"], f"{OUT_DIR}/figures/compare_batch.png", "Batch size (cùng 20 epoch)")
plot_compare([base_runs[0]] + hp_runs[3:], ["train_loss", "val_loss", "val_macro_f1"], f"{OUT_DIR}/figures/compare_hparam.png", "Độ rộng / độ sâu / weight decay / epoch")
display(Image(f"{OUT_DIR}/figures/compare_batch.png", width=800))
display(Image(f"{OUT_DIR}/figures/compare_hparam.png", width=950))

**Đối chiếu (hparam):** _điền sau khi chạy_

### Chủ đề 4 — Dropout (`drop-*`)
**Dự đoán trước:** dropout là thuốc cho quá khớp. Nếu baseline chưa quá khớp (khoảng cách val − train loss nhỏ, val loss vẫn giảm ở epoch 20) thì dropout chỉ làm
mô hình học chậm hơn: q = 0,1 gần như không đổi, q = 0,3 và 0,5 làm giảm macro-F1 (vượt nhiễu), và khoảng cách train–val thu hẹp. Train loss đo ở eval mode nên không bị "phồng" giả do dropout.

In [ ]:
drop_runs = [R({**BASE, "exp_id": f"drop-{q}", "group": "dropout", "dropout": q, "description": f"dropout q={q} sau mỗi ReLU ẩn"}, show=(q == 0.3))
             for q in (0.1, 0.3, 0.5)]
df_drop = table([base_runs[0]] + drop_runs, BASE_F1, NOISE)
for r in [base_runs[0]] + drop_runs:
    s = r["summary"]
    print(f"{r['cfg']['exp_id']:10s} gap val−train loss cuối = {s['final_val_loss'] - s['final_train_loss']:+.4f}")
plot_compare([base_runs[0]] + drop_runs, ["train_loss", "val_loss", "val_macro_f1"], f"{OUT_DIR}/figures/compare_dropout.png", "Dropout")
display(Image(f"{OUT_DIR}/figures/compare_dropout.png", width=950))

**Đối chiếu (dropout):** _điền sau khi chạy_

### Chủ đề 5 — Gradient clipping (`clip-*`)
Chọn `c` **dựa trên grad_norm của baseline**: lấy trung vị của grad_norm trung bình theo epoch của `base-s1`, để clipping thực sự kích hoạt ở một phần đáng kể các bước
(tỉ lệ bước bị cắt được ghi trong `clip_frac`).

**Dự đoán trước:** ở lr baseline, clipping chỉ làm bước nhỏ lại ở các bước gradient lớn → kết quả gần baseline (trong nhiễu), có thể chậm hơn chút.
Ở lr cao (×10, ×30) không clip: loss dao động mạnh hoặc NaN (grad_norm có gai lớn); có clip: chuẩn bước cập nhật bị chặn ở lr·c nên huấn luyện ổn định hơn và F1 cao hơn hẳn bản không clip.

In [ ]:
gn = np.array(base_runs[0]["history"]["grad_norm"])
gmax = np.array(base_runs[0]["history"]["grad_norm_max"])
C = float(f"{np.median(gn):.2g}")
print("base-s1 grad_norm TB theo epoch:", gn.round(3))
print("base-s1 grad_norm max theo epoch:", gmax.round(3))
print("chọn c =", C)
clip_runs = [R({**BASE, "exp_id": f"clip-{C:g}", "group": "clipping", "clip_norm": C, "description": f"clip c={C:g} (trung vị grad_norm baseline), lr baseline"})]
for k in (10, 30):
    clip_runs.append(R({**BASE, "exp_id": f"clip-none-lrx{k}", "group": "clipping", "lr": k * BASE_LR,
                        "description": f"lr x{k}, KHÔNG clip (phản chứng)"}, show=False))
    clip_runs.append(R({**BASE, "exp_id": f"clip-{C:g}-lrx{k}", "group": "clipping", "lr": k * BASE_LR, "clip_norm": C,
                        "description": f"lr x{k}, clip c={C:g}"}, show=(k == 10)))
df_clip = table([base_runs[0]] + clip_runs, BASE_F1, NOISE)
for r in clip_runs:
    print(f"{r['cfg']['exp_id']:16s} tỉ lệ bước bị clip theo epoch:", np.round(r["history"]["clip_frac"], 2).tolist()[:10], "...")
plot_compare([base_runs[0]] + clip_runs, ["val_loss", "val_macro_f1", "grad_norm_max"], f"{OUT_DIR}/figures/compare_clipping.png", "Gradient clipping (lr thường và lr cao)")
display(Image(f"{OUT_DIR}/figures/compare_clipping.png", width=950))

**Đối chiếu (clipping):** _điền sau khi chạy_

### Chủ đề 6 — Mixed precision (`amp-*`)
FP16: `autocast(float16)` + `GradScaler` (unscale trước khi đo/clip gradient). BF16: `autocast(bfloat16)`, không GradScaler. Tham số vẫn FP32.

**Dự đoán trước:** độ chính xác (F1) gần như không đổi so với FP32 (trong nhiễu). Với mạng rất nhỏ (48k tham số) và batch 512, thời gian bị chi phối bởi chi phí gọi kernel
và vòng lặp Python, nên FP16 **có thể không nhanh hơn** (thậm chí chậm hơn vì thêm bước ép kiểu và GradScaler); bộ nhớ cực đại cũng gần như không đổi vì dữ liệu đã nằm trên GPU ở FP32.
BF16 trên T4 (không có phần cứng BF16) sẽ chậm hơn. Với M-wide, FP16 có nhiều cơ hội có lợi hơn.

In [ ]:
print("GPU hỗ trợ BF16:", torch.cuda.is_bf16_supported() if device == "cuda" else "không có GPU")
amp_runs = [R({**BASE, "exp_id": "amp-fp16", "group": "amp", "precision": "fp16", "description": "FP16 autocast + GradScaler"})]
try:
    amp_runs.append(R({**BASE, "exp_id": "amp-bf16", "group": "amp", "precision": "bf16", "description": "BF16 autocast, không GradScaler"}, show=False))
except Exception as e:
    print("BF16 không chạy được trên máy này:", repr(e))
amp_runs.append(R({**BASE, "exp_id": "amp-wide-fp16", "group": "amp", "precision": "fp16", "hidden": (512, 256),
                   "description": "M-wide + FP16 (so với hp-wide FP32)"}, show=False))
hp_wide = [r for r in hp_runs if r["cfg"]["exp_id"] == "hp-wide"][0]
df_amp = table([base_runs[0]] + amp_runs[:-1] + [hp_wide, amp_runs[-1]], BASE_F1, NOISE)
for r in amp_runs:
    print(f"{r['cfg']['exp_id']:14s} số bước bị GradScaler bỏ qua (tổng):", int(np.sum(r["history"]["skipped_steps"])))
plot_compare([base_runs[0]] + amp_runs[:-1], ["val_loss", "val_macro_f1", "epoch_time_s"], f"{OUT_DIR}/figures/compare_amp.png", "FP32 vs FP16 vs BF16")
display(Image(f"{OUT_DIR}/figures/compare_amp.png", width=950))

**Đối chiếu (amp):** _điền sau khi chạy_

### Chủ đề 7 — Khởi tạo tham số (`init-*`)
`xavier` ở đây là `xavier_normal_` (Var = 2/(n_vào+n_ra)); `he` là `kaiming_normal_` (Var = 2/n_vào); `default` là khởi tạo mặc định của `nn.Linear`
(kaiming_uniform với a = √5, tức U(±1/√n_vào), Var = 1/(3·n_vào)) và bias mặc định ≠ 0.

**Dự đoán trước:**
- `zeros`: mọi nơ-ron trong một lớp ẩn có cùng đầu ra 0, ReLU(0) = 0 và đạo hàm tại 0 bằng 0 (theo quy ước PyTorch) → gradient của W1, W2 bằng 0, chỉ bias lớp cuối học được.
  Mạng chỉ học được prior của lớp (đoán lớp đa số) → acc ≈ 0,4876, macro-F1 ≈ 0,094. Loss bước 0 = ln 7 chính xác.
- `normal` (std 0,01): kích hoạt co lại ~0,01·√n mỗi lớp → tín hiệu và gradient rất nhỏ ở bước 0, học chậm lúc đầu nhưng với 3 lớp vẫn thoát được.
- `xavier` và `default`: phương sai nhỏ hơn He (với ReLU, kích hoạt giảm dần qua các lớp), nhưng mạng chỉ 3 lớp nên khác biệt F1 so với He có thể nằm trong nhiễu.
  Để thấy rõ hiện tượng như biểu đồ "30 lớp ReLU" của slide, ta đo thêm std kích hoạt của một mạng 30 lớp (không huấn luyện).

In [ ]:
init_runs = [R({**BASE, "exp_id": f"init-{name}", "group": "init", "init": name, "description": f"khởi tạo {name}, bias=0" if name != "default" else "khởi tạo mặc định nn.Linear"},
               show=(name == "zeros")) for name in ("zeros", "normal", "xavier", "default")]
df_init = table([base_runs[0]] + init_runs, BASE_F1, NOISE)
print("độ lệch chuẩn kích hoạt ở bước 0 (sau ReLU1, ReLU2, logit) trên 4096 mẫu val:")
for r in [base_runs[0]] + init_runs:
    print(f"  {r['cfg']['init']:8s}", np.round(r["summary"]["act_std_step0"], 5).tolist(), f"  loss bước 0 = {r['summary']['step0_loss']:.4f}")
plot_compare([base_runs[0]] + init_runs, ["val_loss", "val_macro_f1", "grad_norm"], f"{OUT_DIR}/figures/compare_init.png", "Khởi tạo tham số")
display(Image(f"{OUT_DIR}/figures/compare_init.png", width=950))

In [ ]:
# Minh hoạ "30 lớp ReLU" (không huấn luyện, không phải một dòng thí nghiệm): std kích hoạt qua 30 lớp ẩn 256 nơ-ron
import torch.nn as nn
from model import init_weights
xb = data["X_val"][:4096]
plt.figure(figsize=(6, 3.6))
for name in ("normal", "xavier", "he", "default"):
    set_seed(0)
    layers = []
    n_in = 54
    for _ in range(30):
        layers += [nn.Linear(n_in, 256), nn.ReLU()]; n_in = 256
    deep = nn.Sequential(*layers).to(device)
    init_weights(deep, name)
    stds, h = [], xb
    with torch.no_grad():
        for layer in deep:
            h = layer(h)
            if isinstance(layer, nn.ReLU):
                stds.append(h.std().item())
    plt.semilogy(range(1, 31), stds, "o-", ms=3, label=name)
    print(f"{name:8s} std lớp 1, 10, 20, 30:", [f"{stds[i]:.2e}" for i in (0, 9, 19, 29)])
plt.xlabel("lớp ẩn"); plt.ylabel("std kích hoạt (log)"); plt.title("Mạng 30 lớp ReLU ở bước 0"); plt.legend(); plt.grid(alpha=.3)
plt.savefig(f"{OUT_DIR}/figures/compare_init_deep30.png", dpi=110, bbox_inches="tight"); plt.show()

**Đối chiếu (init):** _điền sau khi chạy_

### Kết hợp — cấu hình cuối cùng (`final-*`), chọn **chỉ bằng val**
Lấy bộ tối ưu + lr tốt nhất theo val ở chủ đề 2 (nếu nó tốt hơn SGD+momentum), kết hợp với những yếu tố có ích theo val ở chủ đề 3 (M-wide, 40 epoch).
Hai ứng viên: (A) như trên; (B) như A + lịch lr cosine. Chọn ứng viên có val macro-F1 cao hơn, rồi chạy thêm 2 seed để đo nhiễu.

**Dự đoán trước:** cấu hình cuối sẽ vượt baseline rõ rệt (> 2σ), vì baseline chưa hội tụ ở 20 epoch và năng lực của M-base còn hạn chế; cosine giúp giảm dao động ở cuối nên có thể nhỉnh hơn.

In [ ]:
best_overall = max(best_opt.values(), key=lambda r: r["summary"]["val_macro_f1"])
oc = best_overall["cfg"]
print("bộ tối ưu tốt nhất theo val:", oc["exp_id"], oc["optimizer"], oc["lr"], f"F1={best_overall['summary']['val_macro_f1']:.4f}")
FINAL_BASE = {**BASE, "optimizer": oc["optimizer"], "lr": oc["lr"], "weight_decay": oc["weight_decay"],
              "hidden": (512, 256), "epochs": 40, "group": "final"}
cand = [
    R({**FINAL_BASE, "exp_id": "final-A-s1", "description": f"{oc['optimizer']} lr={oc['lr']:g} + M-wide + 40 epoch"}, keep_state=True, show=False),
    R({**FINAL_BASE, "exp_id": "final-B-s1", "scheduler": "cosine", "description": f"{oc['optimizer']} lr={oc['lr']:g} + M-wide + 40 epoch + cosine"}, keep_state=True, show=False),
]
table(cand, BASE_F1, NOISE)
chosen = max(cand, key=lambda r: r["summary"]["val_macro_f1"])
tag = chosen["cfg"]["exp_id"].split("-")[1]
print("cấu hình cuối cùng (theo val):", chosen["cfg"]["exp_id"])
final_runs = [chosen] + [R({**chosen["cfg"], "seed": s, "exp_id": f"final-{tag}-s{s}", "description": chosen["cfg"]["description"] + f", seed {s}"}, show=False)
                         for s in (2, 3)]
df_final = table(final_runs, BASE_F1, NOISE)
ff1 = np.array([r["summary"]["val_macro_f1"] for r in final_runs])
print(f"final val macro-F1: {ff1.mean():.4f} ± {ff1.std(ddof=1):.4f} (3 seed)   vs baseline {BASE_F1:.4f} ± {BASE_SD:.4f}")
plot_compare([base_runs[0]] + cand + final_runs[1:], ["train_loss", "val_loss", "val_macro_f1"], f"{OUT_DIR}/figures/compare_final.png", "Baseline vs cấu hình cuối")
display(Image(f"{OUT_DIR}/figures/compare_final.png", width=950))

**Đối chiếu (final):** _điền sau khi chạy_

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
Cấu hình cuối đã chọn **chỉ bằng val** ở trên. Bây giờ mới dự đoán trên **toàn bộ** eval (eval mode, trọng số của epoch có val loss thấp nhất) và chấm bằng
`scripts/evaluate.py`. Làm cho baseline (`base-s1`, file phụ trong `baseline_eval/`) và cấu hình cuối (seed 1, file nộp). Không chỉnh cấu hình sau bước này.

In [ ]:
def run_eval_script(pred, out):
    p = subprocess.run([sys.executable, "scripts/evaluate.py", "--pred", pred, "--out", out], cwd=REPO_ROOT, capture_output=True, text=True)
    print(p.stdout, p.stderr)
    assert p.returncode == 0
    return json.load(open(out, encoding="utf-8"))

base_eval_dir = Path(OUT_DIR) / "baseline_eval"; base_eval_dir.mkdir(exist_ok=True)
base_s1 = R({**BASE, "seed": 1, "exp_id": "base-s1", "group": "baseline", "description": "Baseline M-base, seed 1"}, keep_state=True, show=False)
final_eval(base_s1["cfg"], base_s1, data, str(base_eval_dir / "predictions_eval_baseline.csv"))
print("===== BASELINE (base-s1) =====")
EVAL_BASE = run_eval_script(str(base_eval_dir / "predictions_eval_baseline.csv"), str(base_eval_dir / "eval_result_baseline.json"))

final_eval(chosen["cfg"], chosen, data, f"{OUT_DIR}/predictions_eval.csv")
print("===== CẤU HÌNH CUỐI (" + chosen["cfg"]["exp_id"] + ") =====")
EVAL_FINAL = run_eval_script(f"{OUT_DIR}/predictions_eval.csv", f"{OUT_DIR}/eval_result.json")
print(f"eval macro-F1: baseline {EVAL_BASE['macro_f1']:.4f} -> final {EVAL_FINAL['macro_f1']:.4f}  (Δ = {EVAL_FINAL['macro_f1'] - EVAL_BASE['macro_f1']:+.4f})")
print(f"val  macro-F1: baseline {base_s1['summary']['val_macro_f1']:.4f} -> final {chosen['summary']['val_macro_f1']:.4f}")

In [ ]:
# Phân tích lỗi theo lớp trên eval
names = ["Spruce/Fir", "Lodgepole Pine", "Ponderosa Pine", "Cottonwood/Willow", "Aspen", "Douglas-fir", "Krummholz"]
rows = []
for tag_, res in (("baseline", EVAL_BASE), ("final", EVAL_FINAL)):
    for pc in res["per_class"]:
        rows.append(dict(model=tag_, cls=pc["cls"], name=names[pc["cls"]], support=pc["support"],
                         precision=pc["precision"], recall=pc["recall"], f1=pc["f1"]))
df_cls = pd.DataFrame(rows).pivot(index=["cls", "name", "support"], columns="model", values=["precision", "recall", "f1"])
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 200):
    display(df_cls)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (tag_, res) in zip(axes, (("baseline base-s1", EVAL_BASE), (f"final {chosen['cfg']['exp_id']}", EVAL_FINAL))):
    cm = np.array(res["confusion_matrix"])
    cmn = cm / cm.sum(1, keepdims=True)
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(7):
        for j in range(7):
            ax.text(j, i, f"{cmn[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center", fontsize=6.5,
                    color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_xticks(range(7)); ax.set_yticks(range(7))
    ax.set_xlabel("dự đoán"); ax.set_ylabel("nhãn thật")
    ax.set_title(f"{tag_}: macro-F1 {res['macro_f1']:.4f}, acc {res['accuracy']:.4f}\n(chuẩn hoá theo hàng = recall)", fontsize=9)
fig.colorbar(im, ax=axes, shrink=0.8)
fig.savefig(f"{OUT_DIR}/figures/compare_eval_confusion.png", dpi=110, bbox_inches="tight"); plt.show()

cm = np.array(EVAL_FINAL["confusion_matrix"])
off = cm.copy(); np.fill_diagonal(off, 0)
f1c = np.array([pc["f1"] for pc in EVAL_FINAL["per_class"]])
hard = int(f1c.argmin())
print(f"lớp khó nhất (final): {hard} ({names[hard]}), F1 = {f1c[hard]:.4f}; hay bị nhầm thành lớp {int(off[hard].argmax())} "
      f"({off[hard].max()} / {cm[hard].sum()} mẫu)")
i, j = np.unravel_index(off.argmax(), off.shape)
print(f"cặp nhầm lẫn nhiều nhất về số lượng: thật {i} -> đoán {j}: {off[i, j]} mẫu")

**Phân tích lỗi:** _điền sau khi chạy_

In [ ]:
# Bảng experiments.xlsx: mỗi exp_id một dòng; eval chỉ cho baseline (base-s1) và cấu hình cuối
GROUP_ORDER = ["baseline", "loss", "optimizer", "hparam", "dropout", "clipping", "amp", "init", "final"]
results = load_results(f"{OUT_DIR}/results")
results.sort(key=lambda r: (GROUP_ORDER.index(r["cfg"]["group"]), r["cfg"]["exp_id"]))
eval_map = {"base-s1": EVAL_BASE, chosen["cfg"]["exp_id"]: EVAL_FINAL}
rows = [to_row(r, eval_map.get(r["cfg"]["exp_id"]),
               notes=("trùng cấu hình base-s1 (kiểm tra tái lập)" if r["cfg"]["exp_id"] == f"opt-sgdm-lr{BASE_LR:g}" else "")
                     + ("cấu hình cuối, file nộp" if r["cfg"]["exp_id"] == chosen["cfg"]["exp_id"] else ""))
        for r in results]

def best_of(group):
    g = [r for r in results if r["cfg"]["group"] == group and r["summary"]["val_macro_f1"] is not None]
    b = max(g, key=lambda r: r["summary"]["val_macro_f1"])
    d = b["summary"]["val_macro_f1"] - BASE_F1
    return f"tốt nhất {b['cfg']['exp_id']} (val F1 {b['summary']['val_macro_f1']:.4f}, Δ {d:+.4f} so với TB baseline, {'vượt' if abs(d) > NOISE else 'trong'} 2σ={NOISE:.4f})"

SUMMARY_NOTES = {g: best_of(g) for g in GROUP_ORDER if any(r["cfg"]["group"] == g for r in results)}
SUMMARY_NOTES["baseline"] = f"3 seed: val F1 {BASE_F1:.4f} ± {BASE_SD:.4f}; lr={BASE_LR:g} chọn bằng val; eval F1 {EVAL_BASE['macro_f1']:.4f}"
SUMMARY_NOTES["final"] = (f"{chosen['cfg']['exp_id']}: val F1 {ff1.mean():.4f} ± {ff1.std(ddof=1):.4f} (3 seed); "
                          f"eval F1 {EVAL_FINAL['macro_f1']:.4f} (baseline {EVAL_BASE['macro_f1']:.4f})")
write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx",
           seed_ids=["base-s1", "base-s2", "base-s3"], summary_notes=SUMMARY_NOTES)
figs = {p.stem for p in Path(f"{OUT_DIR}/figures").glob("*.png") if not p.stem.startswith("compare_")}
ids = {r["exp_id"] for r in rows}
print(len(rows), "dòng trong bảng;", len(figs), "ảnh <exp_id>.png;", "khớp" if figs == ids else f"LỆCH: {figs ^ ids}")
assert figs == ids
for k, v in SUMMARY_NOTES.items():
    print(f"{k:9s} {v}")

In [ ]:
# Đóng gói: chép code hoàn chỉnh vào submission_<MSSV>/code/ (notebook có output được chép sau khi lưu)
import shutil
dst = Path(OUT_DIR) / "code"; dst.mkdir(exist_ok=True)
for f in ("data.py", "model.py", "optimizer.py", "train.py", "plots.py", "results_table.py", "runner.py"):
    shutil.copy(Path(CODE_DIR) / f, dst / f)
print(sorted(p.name for p in dst.iterdir()))
print("Thư mục nộp:", OUT_DIR)
print(sorted(p.name for p in Path(OUT_DIR).iterdir()))